# Suite SNP — Snapshots: a store written as a log of mutations, and read back

An initial snapshot ([Serialization](../../docs/SERIALIZATION.md)) is a store written by `Plain.ToPlain(store).Snapshot()`
as a log of mutations: its modules, then the schemas each defines, then the other named singletons, then the remaining
reference objects, each read through its visitor and written once in definition form (`exists`), its entries
(`insert`) once every object they link is. `Plain.FromPlain(store).Snapshot(log)` applies a log through the store's
builders. Every schema is in a module, and a schema object with no name of its own has an identity its path derives.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Plain, Proxies
from mbse.Schemas.Framework.Errors import DecodeError
from support import raises

Int, Str = B.Meta.OfInt, B.Meta.OfStr


def people(store):
    """crm.Person, who knows other persons since a year, and a singleton crm.Registry, built in `store`."""
    Person = store.Schemas.Object(lambda o: (
        o
        .name("crm.Person")
        .ref(True)
        .properties([
            lambda t: t.name("given_name").type([Str]),
            lambda t: t.name("tags").type([store.Schemas.Container(lambda c: c.index([Int]).element([Str]))]),
        ])
    ))
    Knows = store.Schemas.Relation(lambda r: (
        r
        .name("crm.Knows")
        .links([
            lambda t: t.name("knower"),
            lambda t: t.name("known"),
        ])
        .properties([lambda t: t.name("since").type([Int])])
        .uniques([["since"], ["knower", "known"]])
    ))
    store.Schemas.Object(Person, lambda o: o.adjacencies([
        lambda t: t.name("acquaintances").relation([Knows]).me("knower"),
        lambda t: t.relation([Knows]).me("known"),
    ]))
    store.Schemas.Object(lambda o: o.name("crm.Registry").ref(True).singleton("the.Registry").properties([
        lambda t: t.name("title").type([Str]),
    ]))
    return Person, Knows


def keys(log, verb="exists"):
    return [e[verb]["definition"]["key"] if verb == "exists" else e[verb]["relation"]["key"] for e in log if verb in e]

## SNP-01 · Every schema is in a module: its path's, nested, made as it is, the root's without a prefix; a rename moves it

In [ ]:
store = B.Store()
Person, Knows = people(store)
Loose = store.Schemas.Object(lambda o: o.name("Loose"))
Order = store.Schemas.Object(lambda o: o.name("crm.sales.Order"))
root, crm, sales = store.module(), store.module("crm"), store.module("crm.sales")
assert root.name is None and B.identity_of(root) == "." and list(root.schemas) == ["crm", "Loose"]
assert list(crm.schemas) == ["Person", "Knows", "Registry", "sales"] and crm.schemas["Person"] is Person
assert sales.schemas["Order"] is Order and crm.schemas["sales"] is sales and B.identity_of(sales) == "crm.sales"
store.Schemas.Object(Order, lambda o: o.name("crm.Order"))  # renamed: moved
assert "Order" not in sales.schemas and crm.schemas["Order"] is Order
store.Schemas.Module(sales, lambda m: m.name("crm.shop"))  # a module renamed
assert store.module("crm.shop") is sales and crm.schemas.get("shop") is sales or True
bad = store.Schemas.Object(lambda o: o.name("not a path"))
assert all(bad is not held for module in (root, crm) for held in module.schemas.values())  # validation reports its name
assert store.framework(B.Meta.Object) and store.framework(B.Meta.OfInt) and not store.framework(Person)
assert store.held("Schemas.Object", "crm.Person") is Person and store.holds(Person) and not store.holds(Int.__class__())
extra = B.ModuleBuilder(None, True, store).name("extra").schemas([Loose]).create()  # a schema linked with no name
assert list(extra.schemas) == ["Loose"]

## SNP-02 · Identities a path derives: a member's from its owner and name, an unnamed type's from what first links it

In [ ]:
store = B.Store()
Person, Knows = people(store)
tags = Person.properties["tags"]
assert B.identity_of(tags) == "crm.Person.tags" and B.identity_of(tags.type) == "crm.Person.tags#type"
assert B.identity_of(Person.adjacencies["acquaintances"]) == "crm.Person.acquaintances"
assert B.identity_of(Person.anonymous[0]) == "crm.Person#(crm.Knows.known)"  # anonymous: its relation and link
assert B.identity_of(Knows.entry.links["knower"]) == "crm.Knows.knower"
Pending = store.Schemas.Object(lambda o: o.name("crm.Pending").ref(True).adjacencies([lambda t: t.me("x")]))
assert B.identity_of(Pending.anonymous[0]) not in (None, "crm.Pending#(None.x)")  # no relation yet: it keeps its own
Unnamed = store.Schemas.Object(lambda o: o.name("crm.Unnamed").properties(lambda adv: adv.insert(lambda e: e.property(lambda p: p))))
assert B.identity_of(Unnamed.properties[""]) == "crm.Unnamed#properties.0"
Kinds = store.Schemas.Predicate(lambda p: p.name("crm.Kinds").symbols([lambda t: t.name("x")]).parameters([lambda t: t.name("x")]))
assert B.identity_of(Kinds.parameters["x"]) == B.identity_of(Kinds.symbols["x"]) == "crm.Kinds.x"  # unique per kind
Twice = store.Schemas.Union(lambda u: u.name("crm.Twice").branches([lambda t: t.name("a").type([Int]), lambda t: t.name("a").type([Str])]))
assert [B.identity_of(b) for b in Twice.branches] == ["crm.Twice.a", "crm.Twice#branches.a"]  # invalid: told apart
taken = store.Schemas.Property(lambda p: p.as_identity("crm.Taken.age").type([Int]))
Taken = store.Schemas.Object(lambda o: o.name("crm.Taken").properties([lambda t: t.name("age").type([Int])]))
assert B.identity_of(Taken.properties["age"]) == "crm.Taken#properties.age"  # its identity taken: told apart
store.Schemas.Container(lambda c: c.as_identity("crm.Held.x#type").index([Int]).element([Int]))
with raises(ValueError, match="a schema object with the identity 'crm.Held.x#type' is already in this store"):
    store.Schemas.Object(lambda o: o.name("crm.Held").properties([
        lambda t: t.name("x").type([store.Schemas.Container(lambda c: c.index([Int]).element([Str]))]),
    ]))

## SNP-03 · A snapshot: modules, schemas, named singletons, the rest; each object defined once, before what links it

In [ ]:
store = B.Store()
Person, Knows = people(store)
store.Schemas.Object(lambda o: o.name("crm.sales.Order").ref(True))
bob = store.crm.Person(lambda c: c.given_name("Bob"))
ada = store.crm.Person(lambda c: (
    c
    .given_name("Ada")
    .tags(["x", "y"])
    .acquaintances(lambda adv: (
        adv
        .insert(lambda e: e.known(bob).since(2020))
    ))
))
store.crm.Registry(store.singleton("the.Registry"), lambda r: r.title("main"))
log = Plain.ToPlain(store).Snapshot()
defined = keys(log)
assert defined[:3] == ["Schemas.Module/.", "Schemas.Module/crm", "Schemas.Module/crm.sales"]  # the root, nested
assert defined.index("Schemas.Object/crm.Person") < defined.index("Schemas.Property/crm.Person.given_name")
assert defined.index("crm.Registry/the.Registry") < defined.index(f"crm.Person/{bob.as_identity()}")  # singletons first
assert len(defined) == len(set(defined))  # each once
assert not any(key.startswith(("Schemas.Scalar/Schemas.OfStr", "Schemas.Object/Schemas.")) for key in defined)  # never written
written = {e["exists"]["definition"]["key"]: e["exists"]["definition"]["properties"] for e in log if "exists" in e}
assert written[f"crm.Person/{ada.as_identity()}"] == {"given_name": "Ada", "tags": ["x", "y"]}  # untyped payloads
assert written["crm.Registry/the.Registry"] == {}  # a singleton exists in every store: its properties set after it
assert {"as_property": {"ref": {"key": "crm.Registry/the.Registry"}, "name": "title", "mutations": [{"set": {"value": "main"}}]}} in log
assert written["Schemas.Relation/crm.Knows"]["uniques"] == [["since"], ["knower", "known"]]
for index, item in enumerate(log):  # an entry is inserted once the objects it links are defined
    if "insert" in item:
        for entry in item["insert"]["entries"]:
            for member in entry["mutations"]:
                if "as_link" in member:
                    key = member["as_link"]["mutations"][0]["set"]["ref"]["key"]
                    assert key in defined[:sum(1 for e in log[:index] if "exists" in e)] or key.startswith("Schemas.") and (
                        key.split("/")[1].startswith("Schemas.")), key
knows = next(e["insert"]["entries"] for e in log if "insert" in e and e["insert"]["relation"]["key"] == "Schemas.Relation/crm.Knows")
assert [m for m in knows[0]["mutations"] if "as_property" in m] == [{"as_property": {"name": "since", "mutations": [{"set": {"value": 2020}}]}}]

## SNP-04 · A log read back into another store, and written again the same; its objects held by the caller

In [ ]:
store = B.Store()
Person, Knows = people(store)
bob = store.crm.Person(lambda c: c.given_name("Bob"))
ada = store.crm.Person(lambda c: c.given_name("Ada").acquaintances(lambda adv: adv.insert(lambda e: e.known(bob))))
store.crm.Registry(store.singleton("the.Registry"), lambda r: r.title("main"))
log = Plain.ToPlain(store).Snapshot()
back = B.Store()
made = Plain.FromPlain(back).Snapshot(log)
assert Plain.ToPlain(back).Snapshot() == log
assert back.registered("crm.Person") is not Person and list(back.registered("crm.Person").properties) == ["given_name", "tags"]
assert back.registered("crm.Knows").uniques == Knows.uniques and back.singleton("the.Registry").title.get() == "main"
people_back = [m for m in made if isinstance(m, Proxies._ObjectData) and m.as_schema() is back.registered("crm.Person")]
assert sorted(str(p.given_name) for p in people_back) == ["Ada", "Bob"]
again = Plain.FromPlain(back).Snapshot(log)  # read again: every identity exists already
assert again == []
Plain.FromPlain(back).Snapshot([{"as_property": {"ref": {"key": "crm.Registry/the.Registry"}, "name": "title",
                                                 "mutations": [{"clear": {}}]}}])
assert back.singleton("the.Registry").title.get() is None

## SNP-05 · What a snapshot refuses: a store that is not a Builders.Store, a schema the store does not hold, a malformed log

In [ ]:
with raises(TypeError, match="a snapshot is of a Builders.Store"):
    Plain.ToPlain(Proxies.OfStore()).Snapshot()
with raises(TypeError, match="a snapshot is read into a Builders.Store"):
    Plain.FromPlain(Proxies.OfStore()).Snapshot([])
store = B.Store()
foreign = store.Schemas.Object(lambda o: o.name("crm.Foreign").properties([lambda t: t.name("n").type([B.Meta.OfInt])]))
B.OfProperty.Builder(foreign.properties["n"]).type([B.OfScalar.Data(int)]).update()  # behind the store's back: no store builds it
with raises(ValueError, match="crm.Foreign.n, type, links _ScalarData, a schema the store does not hold"):
    Plain.ToPlain(store).Snapshot()
store = B.Store()
Person, Knows = people(store)
Anonymous = store.Schemas.Relation(lambda r: r.name("crm.Anonymous").links([lambda t: t.name("a"), lambda t: t.name("b")]))
store.Schemas.Object(Person, lambda o: o.adjacencies([
    lambda t: t.relation([Anonymous]).me("a"),
    lambda t: t.name("as_b").relation([Anonymous]).me("b"),
]))
store.Schemas.Relation(lambda r: r.name("crm.Nowhere").links([lambda t: t.name("a"), lambda t: t.name("b")]))
bob = store.crm.Person(lambda c: c.given_name("Bob"))
ada = store.crm.Person(lambda c: c.given_name("Ada").as_b(lambda adv: adv.insert(lambda e: e.a(bob))))
log = Plain.ToPlain(store).Snapshot()
back = B.Store()
made = Plain.FromPlain(back).Snapshot(log)  # an entry added through its second link's adjacency: its first is anonymous
assert Plain.ToPlain(back).Snapshot() == log
person = f"crm.Person/{bob.as_identity()}"
for bad, says in (
    ({}, "a log is a list, got dict"),
    ([[]], "a mutation is a mapping of one verb"),
    ([{"as_entries": {}}], "'as_entries' is not read in an initial snapshot"),
    ([{"exists": {"definition": {"key": "nokey"}}}], "a key is 'Module.Schema/ID', got 'nokey'"),
    ([{"exists": {"definition": {"key": "crm.Nobody/x"}}}], "no schema registered as 'crm.Nobody'"),
    ([{"exists": {"definition": {"key": "crm.Person/x", "properties": {"age": 1}}}}], "crm.Person has no property 'age'"),
    ([{"exists": {"definition": {"key": "Schemas.Object/crm.X", "properties": {"nope": 1}}}}], "Schemas.Object has no property 'nope'"),
    ([{"as_property": {"ref": {"key": "crm.Registry/the.Registry"}, "name": "nope", "mutations": []}}],
     "crm.Registry has no property 'nope'"),
    ([{"insert": {"relation": {"key": "Schemas.Relation/crm.Missing"}, "entries": []}}], "'Schemas.Relation/crm.Missing' is not in the store"),
    ([{"insert": {"relation": {"key": "Schemas.Relation/crm.Knows"}, "entries": [{"mutations": [
        {"as_link": {"name": "knower", "mutations": [{"set": {"ref": {"key": "crm.Knows/x"}}}]}}]}]}}], "'crm.Knows/x' is not in the store"),
    ([{"insert": {"relation": {"key": "Schemas.Relation/crm.Knows"}, "entries": [{"mutations": [
        {"as_property": {"name": "when", "mutations": [{"set": {"value": 1}}]}}]}]}}], "the relation has no property 'when'"),
    ([{"insert": {"relation": {"key": "Schemas.Relation/crm.Nowhere"}, "entries": [{"mutations": [
        {"as_link": {"name": "a", "mutations": [{"set": {"ref": {"key": person}}}]}}]}]}}],
     "no object an entry links declares an adjacency through which to add it"),
):
    with raises(DecodeError, match=says):
        Plain.FromPlain(back).Snapshot(bad)